In [1]:
from pyspark.sql.types import *
from pyspark.sql import *
from pyspark.sql.functions import * 

In [2]:
spark = SparkSession.builder.appName("Pizza Sales Analytics").getOrCreate()

In [3]:
# we will not use inferSchema in the next steps, we will define the schema for each dataframe and load the data again. This is because inferSchema can sometimes infer the wrong datatype for a column, which can lead to errors later on when we try to perform operations on the data. By defining the schema ourselves, we can ensure that the data is loaded correctly and that we can perform the necessary operations without any issues.
schema_order_details = StructType([StructField('order_details_id', IntegerType(), False),
                            StructField('order_id', IntegerType(), False),
                            StructField('pizza_id', StringType(), False),
                            StructField('quantity', IntegerType(), False)])

schema_orders = StructType([StructField('order_id', IntegerType(), False),
                            StructField('date', DateType(), False),
                            StructField('time', TimestampType(), False)])

schema_pizza_types = StructType([StructField('pizza_type_id', StringType(), False),
                                  StructField('name', StringType(), False),
                                  StructField('category', StringType(), False),
                                  StructField('ingredients', StringType(), False)])

schema_pizzas = StructType([StructField('pizza_id', StringType(), False),
                            StructField('pizza_type_id', StringType(), False),
                            StructField('size', StringType(), False),
                            StructField('price', DoubleType(), False)])

In [4]:
# Load the pizza sales data from a CSV file
order_details_df = spark.read.csv('../data/order_details.csv', header=True, schema= schema_order_details)

pizza_types_df = spark.read.csv('../data/pizza_types.csv', header=True, schema = schema_pizza_types) 

orders_df = spark.read.csv('../data/orders.csv', header=True,  schema = schema_orders)

pizzas_df = spark.read.csv('../data/pizzas.csv', header=True, schema = schema_pizzas)

In [5]:
# Check the datatype of the columns of all dataframes
orders_df.printSchema()
pizza_types_df.printSchema()
order_details_df.printSchema()
pizzas_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- time: timestamp (nullable = true)

root
 |-- pizza_type_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- ingredients: string (nullable = true)

root
 |-- order_details_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- pizza_id: string (nullable = true)
 |-- quantity: integer (nullable = true)

root
 |-- pizza_id: string (nullable = true)
 |-- pizza_type_id: string (nullable = true)
 |-- size: string (nullable = true)
 |-- price: double (nullable = true)



In [6]:
# joining all the dataframers together
combined_df = order_details_df \
                .join(pizzas_df, 'pizza_id') \
                .join(orders_df, 'order_id') \
                .join(pizza_types_df, 'pizza_type_id')

In [7]:
# Getting the total count of records after joining all the tables.
combined_df.show()

+-------------+--------+--------------+----------------+--------+----+-----+----------+-------------------+--------------------+--------+--------------------+
|pizza_type_id|order_id|      pizza_id|order_details_id|quantity|size|price|      date|               time|                name|category|         ingredients|
+-------------+--------+--------------+----------------+--------+----+-----+----------+-------------------+--------------------+--------+--------------------+
|     hawaiian|       1|    hawaiian_m|               1|       1|   M|13.25|2015-01-01|2026-03-27 11:38:36|  The Hawaiian Pizza| Classic|Sliced Ham, Pinea...|
|  classic_dlx|       2| classic_dlx_m|               2|       1|   M| 16.0|2015-01-01|2026-03-27 11:57:40|The Classic Delux...| Classic|Pepperoni, Mushro...|
|  five_cheese|       2| five_cheese_l|               3|       1|   L| 18.5|2015-01-01|2026-03-27 11:57:40|The Five Cheese P...|  Veggie|Mozzarella Cheese...|
|    ital_supr|       2|   ital_supr_l|       

In [8]:
# performing aggregation on timestamp column to categorize
# the order time into Morning, Afternoon, Night.
# Morning - 08:00:00 - 11:59:59
# Afternoon - 12:00:00 - 15:59:59
# Evening - 16:00:00 - 19:59:59
# Night - 20:00:00 - 23:59:59
# we need to create new column called hour to extract the hour 
# from the time column and then perform aggregation on the new column
daytime_analysis_df = combined_df.withColumn("hour", hour(col("time")))

In [9]:
# now we got the new df with hour column now we can perform the aggregation
daytime_analysis_df = daytime_analysis_df.withColumn('timeofday', 
                         when ((col('hour') >= 6) & (col("hour") < 12 ), 'Morning')
                        .when ((col('hour') >= 12) & (col("hour") < 16 ), 'Afternoon')
                        .when ((col('hour') >= 16) & (col("hour") < 20 ), 'Evening')
                        .otherwise('Night') )

In [10]:
daytime_analysis_df.filter(col('timeofday') == 'Afternoon').show()

+-------------+--------+--------------+----------------+--------+----+-----+----------+-------------------+--------------------+--------+--------------------+----+---------+
|pizza_type_id|order_id|      pizza_id|order_details_id|quantity|size|price|      date|               time|                name|category|         ingredients|hour|timeofday|
+-------------+--------+--------------+----------------+--------+----+-----+----------+-------------------+--------------------+--------+--------------------+----+---------+
|    ital_supr|       3|   ital_supr_m|               7|       1|   M| 16.5|2015-01-01|2026-03-27 12:12:28|The Italian Supre...| Supreme|Calabrese Salami,...|  12|Afternoon|
|   prsc_argla|       3|  prsc_argla_l|               8|       1|   L|20.75|2015-01-01|2026-03-27 12:12:28|The Prosciutto an...| Supreme|Prosciutto di San...|  12|Afternoon|
|    ital_supr|       4|   ital_supr_m|               9|       1|   M| 16.5|2015-01-01|2026-03-27 12:16:31|The Italian Supre...| S

In [11]:
# Calculate the total revenue generated

combined_df.agg(sum('price').alias('total_revenue')).show()


+-----------------+
|    total_revenue|
+-----------------+
|801944.6999999925|
+-----------------+



In [12]:
# Calculate total number of orders

combined_df.agg(max('order_id').alias('total_orders')).show()


+------------+
|total_orders|
+------------+
|       21350|
+------------+



In [13]:
# Calculate the average order value

total_order_price = combined_df.groupBy('order_id') \
    .agg(sum('price').alias('total_price'))

total_order_price.agg(avg('total_price')).show()

+----------------+
|avg(total_price)|
+----------------+
|37.5618126463697|
+----------------+



In [14]:
# Total Pizzas Sold

combined_df.agg(count('pizza_id').alias('total_pizzas_sold')).show()

+-----------------+
|total_pizzas_sold|
+-----------------+
|            48620|
+-----------------+



In [15]:
# Top 10 most sold pizza types

combined_df.groupBy('pizza_type_id').count().orderBy(desc('count')).limit(10).show()

+-------------+-----+
|pizza_type_id|count|
+-------------+-----+
|  classic_dlx| 2416|
|      bbq_ckn| 2372|
|     hawaiian| 2370|
|    pepperoni| 2369|
|     thai_ckn| 2315|
|     cali_ckn| 2302|
|     sicilian| 1887|
|   spicy_ital| 1887|
|   southw_ckn| 1885|
|  four_cheese| 1850|
+-------------+-----+



In [16]:
# Least Popular Pizza Types

combined_df.groupBy('pizza_type_id').count().orderBy(('count')).limit(5).show()

+-------------+-----+
|pizza_type_id|count|
+-------------+-----+
|   brie_carre|  480|
| mediterraneo|  923|
|    calabrese|  927|
| spinach_supr|  940|
|   spin_pesto|  957|
+-------------+-----+



In [17]:
# Daily Revenue Trend
# To calculate total revenue generated in a day. So we have to group order date 
# and sum the total revenue in that timeframe.

revenue_generated_by_day = combined_df.groupBy('date').agg(sum('price').alias('revenue_per_day'))

revenue_generated_by_day.show()

+----------+------------------+
|      date|   revenue_per_day|
+----------+------------------+
|2015-03-09|            2301.3|
|2015-05-19|           1918.25|
|2015-03-06|           2407.55|
|2015-04-09|2027.0500000000002|
|2015-09-02|1848.8000000000002|
|2015-12-22|1834.7000000000003|
|2015-05-10|2275.8500000000004|
|2015-09-28|2002.0500000000002|
|2015-03-12|1927.6500000000005|
|2015-03-16|           2274.55|
|2015-04-01|2149.6000000000004|
|2015-04-24|2900.2000000000003|
|2015-03-11|           2174.95|
|2015-06-15|2561.1000000000004|
|2015-10-16|            2467.9|
|2015-11-02|           2281.15|
|2015-11-27| 4342.699999999999|
|2015-12-31|           2802.75|
|2015-08-01|           2377.05|
|2015-04-19|            1470.2|
+----------+------------------+
only showing top 20 rows


In [ ]:
# Monthly revenue 
# find out monthly revenue also get the best revenue month


root
 |-- pizza_type_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- pizza_id: string (nullable = true)
 |-- order_details_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- size: string (nullable = true)
 |-- price: double (nullable = true)
 |-- date: date (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- ingredients: string (nullable = true)



In [ ]:
# Orders per hour
